# 04c - ML Forecasting (XGBoost/LightGBM)

**Objective**: Forecast using machine learning models

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
print('Libraries imported')

Libraries imported


In [2]:
# Load feature-engineered data
try:
    df = pd.read_csv('../data/clean/maize_features_ml.csv', parse_dates=['date'])
except FileNotFoundError:
    try:
        df = pd.read_csv('data/clean/maize_features_ml.csv', parse_dates=['date'])
    except FileNotFoundError:
        # Fallback to regular features
        df = pd.read_csv('../data/clean/maize_features.csv', parse_dates=['date'])

print(f'Dataset: {df.shape}')
df.head()


Dataset: (168, 26)


,date,price,observations,year,month,quarter,day_of_year,month_sin,month_cos,price_lag_1m,...,price_rolling_std_6m,price_rolling_mean_12m,price_rolling_std_12m,inflation_mom,inflation_yoy,season,season_harvest_long,season_harvest_short,season_lean,season_normal
0,2007-01-01,21.00,4,2007,1,1,1,0.500000,8.660254e-01,24.00,...,1.231530,23.416667,1.088855,-12.500000,-10.638298,lean,False,False,True,False
1,2007-02-01,21.50,4,2007,2,1,32,0.866025,5.000000e-01,21.00,...,1.382178,23.187500,1.182856,2.380952,-11.340206,lean,False,False,True,False
2,2007-03-01,18.50,4,2007,3,1,60,1.000000,6.123234e-17,21.50,...,2.245829,22.770833,1.788404,-13.953488,-21.276596,lean,False,False,True,False
3,2007-04-01,19.25,4,2007,4,2,91,0.866025,-5.000000e-01,18.50,...,2.431135,22.270833,1.869183,4.054054,-23.762376,harvest_long,True,False,False,False
4,2007-05-01,21.25,4,2007,5,2,121,0.500000,-8.660254e-01,19.25,...,1.927866,22.062500,1.828204,10.389610,-10.526316,harvest_long,True,False,False,False


In [3]:
# Select features
feature_cols = [c for c in df.columns if c not in ['date', 'price', 'observations', 'season']]
X = df[feature_cols]
y = df['price']
print(f'Features: {len(feature_cols)}')
print(feature_cols)

Features: 22
['year', 'month', 'quarter', 'day_of_year', 'month_sin', 'month_cos', 'price_lag_1m', 'price_lag_3m', 'price_lag_6m', 'price_lag_12m', 'price_rolling_mean_3m', 'price_rolling_std_3m', 'price_rolling_mean_6m', 'price_rolling_std_6m', 'price_rolling_mean_12m', 'price_rolling_std_12m', 'inflation_mom', 'inflation_yoy', 'season_harvest_long', 'season_harvest_short', 'season_lean', 'season_normal']


In [4]:
# Train-test split
train_size = len(df) - 12
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

Train: 156, Test: 12


In [5]:
# Train XGBoost
xgb_model = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mape = np.mean(np.abs((y_test - xgb_pred) / y_test)) * 100
print(f'XGBoost - MAE: {xgb_mae:.2f}, RMSE: {xgb_rmse:.2f}, MAPE: {xgb_mape:.2f}%')

XGBoost - MAE: 1.17, RMSE: 1.77, MAPE: 2.10%


In [6]:
# Train LightGBM
lgbm_model = LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbose=-1)
lgbm_model.fit(X_train, y_train)
lgbm_pred = lgbm_model.predict(X_test)
lgbm_mae = mean_absolute_error(y_test, lgbm_pred)
lgbm_rmse = np.sqrt(mean_squared_error(y_test, lgbm_pred))
lgbm_mape = np.mean(np.abs((y_test - lgbm_pred) / y_test)) * 100
print(f'LightGBM - MAE: {lgbm_mae:.2f}, RMSE: {lgbm_rmse:.2f}, MAPE: {lgbm_mape:.2f}%')

LightGBM - MAE: 1.14, RMSE: 1.56, MAPE: 2.07%


In [7]:
from pathlib import Path
Path('../models/trained').mkdir(parents=True, exist_ok=True)

# Save best model (XGBoost)
import joblib
joblib.dump(xgb_model, '../models/trained/xgboost_maize_model.pkl')
forecast_df = pd.DataFrame({'date': df.iloc[train_size:]['date'], 'forecast': xgb_pred})
forecast_df.to_csv('../models/trained/xgboost_forecast.csv', index=False)
print('Models saved')


Models saved
